# VORTEX-AI / CrisisBench: Full-Scale Neural Baselines & Remaining Queue on Kaggle

1. Turn the GPU on for GPU shards (Kaggle: Settings > Accelerator > GPU T4 x2).
2. Mount `brxkeaaryan/vortex-crisisbench-bundle` (or upload `vortex_bundle.zip`).
3. Run all cells. Completed runs are packed continuously into `/kaggle/working/fullscale_results.zip`.

In [ ]:
import glob, importlib, os, shutil, subprocess, sys, zipfile
import torch

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}:', torch.cuda.get_device_name(i))

REQUIRED = {
    'torch_geometric': 'torch-geometric',
    'torchsde': 'torchsde',
    'arch': 'arch',
    'hmmlearn': 'hmmlearn',
    'tabulate': 'tabulate',
}
missing = []
for mod, pkg in REQUIRED.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg)
if missing:
    print('Installing missing packages:', missing, flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
else:
    print('All required packages present.')

In [ ]:
KAGGLE = os.path.isdir('/kaggle/working')
OUT_DIR = '/kaggle/working' if KAGGLE else '/content'
STAGE_DIR = '/tmp/vortex_stage' if KAGGLE else '/content/vortex_stage'
if os.path.exists(STAGE_DIR):
    shutil.rmtree(STAGE_DIR)
os.makedirs(STAGE_DIR, exist_ok=True)

def find_file(name):
    hits = glob.glob('/kaggle/input/**/' + name, recursive=True) + glob.glob(os.path.join(OUT_DIR, name))
    return hits[0] if hits else None

tars = glob.glob('/kaggle/input/**/vortex_bundle.tar', recursive=True)
zips = glob.glob('/kaggle/input/**/vortex_bundle.zip', recursive=True) + glob.glob(os.path.join(OUT_DIR, 'vortex_bundle.zip'))
unz = sorted({os.path.dirname(p) for p in glob.glob('/kaggle/input/**/crisisbench/queue.py', recursive=True)}, key=len)

if tars:
    print('Extracting tar:', tars[0])
    subprocess.run(['tar', '-xf', tars[0], '-C', STAGE_DIR], check=True)
elif zips:
    print('Extracting zip:', zips[0])
    zipfile.ZipFile(zips[0]).extractall(STAGE_DIR)
elif unz:
    src_root = os.path.dirname(unz[0])
    print('Copying unpacked tree:', src_root)
    shutil.copytree(src_root, os.path.join(STAGE_DIR, 'vortex'), dirs_exist_ok=True)
else:
    from google.colab import files
    for name in files.upload():
        if name.startswith('vortex_bundle'):
            zipfile.ZipFile(name).extractall(STAGE_DIR)

REPO = os.path.join(STAGE_DIR, 'vortex')
if not os.path.isdir(os.path.join(REPO, 'crisisbench')):
    alt = glob.glob(os.path.join(STAGE_DIR, '**/crisisbench'), recursive=True)
    assert alt, f'crisisbench not found under {STAGE_DIR}'
    REPO = os.path.dirname(alt[0])

prev = find_file('fullscale_results.zip')
if prev:
    print('Merging previous results zip:', prev)
    zipfile.ZipFile(prev).extractall(REPO)

os.chdir(REPO)
print('Working repo:', os.getcwd(), sorted(os.listdir(REPO)))

In [ ]:
SHARD_MODE = 'all'
WORKERS = 4
THREADS = 1
HOURS = 11.0 if KAGGLE else 3.5
OUT_ZIP = os.path.join(OUT_DIR, 'fullscale_results.zip')

cmd = [
    sys.executable, '-m', 'cloud.kaggle_runner',
    '--mode', SHARD_MODE,
    '--workers', str(WORKERS),
    '--threads', str(THREADS),
    '--hours', str(HOURS),
    '--out', OUT_ZIP,
]
print('$', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=True, cwd=REPO)

In [ ]:
if os.path.exists(OUT_ZIP):
    print(f'Output bundle ready: {OUT_ZIP} ({os.path.getsize(OUT_ZIP) / 1e6:.2f} MB)')
    if not KAGGLE:
        from google.colab import files
        files.download(OUT_ZIP)
else:
    print('No output zip produced.')